In [4]:
import sys
sys.path.insert(0, '../../style_generation_pipeline')

In [5]:
import pandas as pd
import json
import sklearn
import glob
import pickle
from sklearn.model_selection import train_test_split

pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [6]:
from data import *

In [7]:
path='/mnt/swordfish-pool2/milad/hiatus-data/explainability_all_data/'

In [8]:
phase1_docs = pd.read_json(path_or_buf='/mnt/swordfish-pool2/milad/hiatus-data/phase_1/explainability/training_candidates_and_queries.jsonl', lines=True)
phase2_docs = pd.read_json(path_or_buf='/mnt/swordfish-pool2/milad/hiatus-data/phase_2/explainability/all_documents_in_cross_genre.jsonl', lines=True)
all_docs = pd.concat([phase1_docs, phase2_docs]).reset_index()

In [9]:
sample_docs = all_docs.sample(50)
document_id_to_text = {x[0]: x[1] for x in zip(sample_docs.documentID.tolist(), sample_docs.fullText.tolist())}

In [10]:
llm_based_df = pd.read_csv(path + '/refined_and_aggregated_features_final_manually_processed_min_freq_10.csv')
df = pd.read_csv(path +'/refined_and_aggregated_features_final_manually_processed_min_freq_10_filled.csv')
df['low_lvl_and_verifiable'] = df.apply(lambda row: row['Level'] == 'Low' and row['Verifiability'] == 'Yes', axis=1)
accepted_feats = df[df.Verifiability == 'Yes']['final_attribute_name_manually_processed'].tolist()
llm_based_df_filtered = llm_based_df[llm_based_df.final_attribute_name_manually_processed.isin(accepted_feats)]

In [11]:
# Extract a sample of documents and features assigned to them
sample_docs = [(x[1], set(llm_based_df_filtered[llm_based_df_filtered.documentID == x[0]]['original_attribute_name'].tolist())) for x in document_id_to_text.items()]

In [19]:
data = []
for item in sample_docs:
    feats_list = list(item[1])
    data.append([item[0], feats_list[0]])
    for feat in feats_list[1:]:
        data.append(['', feat])
evaluation_csv_df = pd.DataFrame(data, columns=['text', 'feat'])

In [20]:
evaluation_csv_df.head()

,text,feat
0,"Dial M for Murder\n\n“in stories, things turn out the way the author wants. and in real life, they don’t.. always.” ₊˚⊹\n— for some reason, in my head <PERSON> was in this movie haha. i always thought that <PERSON> was a sadistic due to how poorly he treated the actresses in his films but now i also noticed that he also likes to put strangling plots in his movies, so random.\n\nsuch a good thriller ♥︎ i loved everything about it, the dialogues, the characters, the aesthetics! it was very entertaining, nothing feels rushed and makes you think that <PERSON> will succeed, but then he is exposed in such a perfect way that you can see the attention to details & perfectionism <PERSON> had.\n\nwhat i find interesting is that you don’t hate <PERSON>, cause this movie makes his motives understandable and how everything would have succeed if all went along as what he planed, but his mistake was that his plan depended too much on what other people would do, and that you cannot control.",The author employs a mix of declarative and exclamatory sentences to create a sense of excitement and enthusiasm.
1,,"The author employs colloquial expressions, like ""for some reason"", to convey a sense of spontaneity."
2,"Since your civilization is set back to hunter-gatherer, I think we can safely assume that the population is orders of magnitudes smaller than todays. This means that there can be a significant amount of structures left, but the population is nowhere near them, usually.\n\n To make this more plausible you might need reasons on why they keep away from previously inhabited areas (after all, people built megacities there because its a nice place to live). Reasons can be\n\n * Radiation from a nuclear war. Although people can not explain what it is, they get sick, so there are huge forbidden zones.\n * They are deserts now, and the only area where enough plants grow and animals live are near the poles, areas that are today rather sparsely inhabited.\n * Civilization has been buried under lots of ice, e.g. due to earths rotation axis/orientation being shifted, and what previously was hot is now cold and vv.\n * Tectonic movement forced lots of the coastline cities to submerge, while at other places huge amounts of ocean floor are now landmasses. This can even go so far as that the whole earth is just lots of islands now, that were previously mountains (where population density isn't that big)\n\n Also depending on the timeframe, you may consider fast growing plants (that might even have to do something with the initial cataclysm) grow over every building, and cover it; do that for a few decades or centuries, and you have a jungle on top of lots of hills, and only if someone digs a few meters deep, they would discover that the hills are really buildings. But there is probably no reason for your people to do that, so they will never notice, and for them it is a world without ruins.\n\n Also note that your people are probably around in that area for their whole life, so they won't probably go there and ""hey, there is a ruin, lets explore that"". It has been there the whole time, so it might be mentioned but I would not see a need for them to be a major part of the story. Take for example the via appica. In the past hundreds of years, not many living there would have said ""hey, look, an ancient street, lets explore it and its origins"". People had a life to live and probably cared much more about how to get food than what they are currently standing on.","The author employs a range of clause types, including independent and dependent clauses."
3,,"The author uses a variety of sentence structures, including simple, compound, and complex sentences."
4,,"The author uses a range of sentence types, including declarative, interrogative, and imperative sentences."


In [21]:
evaluation_csv_df.to_csv('./sample_of_documents_and_llm_extracted_feats.csv')